# Week 1: Python for data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GDG-TUM/AI-ML-Track-GDG-on-Campus-TUM/blob/main/weekly-sessions/week-01-welcome-and-python-for-data/python_for_data.ipynb)

**GDG on Campus TUM · AI/ML Track**

Before a machine can learn from data, *you* have to be able to look at data. This notebook gives you the three tools every ML engineer uses every day:

| Tool | What it is for |
|------|----------------|
| **NumPy** | Fast arrays and maths |
| **pandas** | Tables of data (like Excel, but programmable) |
| **matplotlib** | Charts |

**How to use this notebook:** click a code cell and press **Shift + Enter** to run it. Run the cells from top to bottom. Break things. That is how you learn.

> The data below is **made up** (synthetic). We generate it ourselves so that we know exactly what pattern is hiding inside it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)  # the seed makes our "random" numbers repeatable
print("NumPy", np.__version__, "| pandas", pd.__version__)

## 1. NumPy: arrays you can do maths on

A Python list stores things. A NumPy **array** lets you do maths on *all* the things at once. No loops needed.

In [ ]:
marks = np.array([54, 71, 88, 62, 95, 47, 79])

print("mean   :", marks.mean())
print("std    :", marks.std().round(2))
print("max    :", marks.max())
print("curved :", marks + 5)  # adds 5 to every element at once
print("passed :", marks >= 50)  # a True/False for every element
print("how many passed:", (marks >= 50).sum())

**Vectorisation** is the big idea: write one expression, let NumPy do the looping in fast compiled code. Machine learning is mostly vectorised maths, so this habit pays off for the rest of the track.

A 2D array is a table of numbers (rows and columns). In ML we call it a **feature matrix**: one row per example, one column per measurement.

In [ ]:
X = np.array(
    [
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
    ]
)
print("shape:", X.shape, "(2 rows, 3 columns)")
print("column means:", X.mean(axis=0))
print("row sums    :", X.sum(axis=1))

## 2. pandas: a table with names

Imagine a campus cafeteria that records, every weekday, the weather, whether it is exam week, and how many cups of chai it sold. We will simulate 120 weekdays.

We *secretly* build in these rules, and then see whether we can find them again by looking at the data:

* rainy days sell **more** chai
* hotter days sell **less** chai
* exam weeks sell **more** chai
* plus random noise, because real life is never perfect

In [ ]:
n_days = 120
dates = pd.date_range("2026-01-05", periods=n_days, freq="B")  # "B" = business days

temperature_c = 29 + 2.5 * np.sin(np.arange(n_days) / 12) + rng.normal(0, 1.0, n_days)
rain_mm = rng.exponential(scale=6, size=n_days) * (rng.random(n_days) < 0.35)
week = np.arange(n_days) // 5
exam_week = np.isin(week, [7, 8, 15, 16, 23])

chai_cups = (
    180 + 3.2 * rain_mm - 4.0 * (temperature_c - 29) + 45 * exam_week + rng.normal(0, 12, n_days)
).round()

df = pd.DataFrame(
    {
        "date": dates,
        "temperature_c": temperature_c.round(1),
        "rain_mm": rain_mm.round(1),
        "exam_week": exam_week,
        "chai_cups": chai_cups.astype(int),
    }
)
df.head()

In [ ]:
print(df.shape, "-> (rows, columns)")
df.describe().round(1)

### Asking questions of the data

Good data work is mostly asking clear questions. Each question below is one line of pandas.

In [ ]:
# Q1: Is chai really different in exam weeks?
print(df.groupby("exam_week")["chai_cups"].mean().round(1))

In [ ]:
# Q2: Which single day sold the most?
df.sort_values("chai_cups", ascending=False).head(3)

In [ ]:
# Q3: How strongly does each measurement move together with chai sales?
df[["temperature_c", "rain_mm", "chai_cups"]].corr().round(2)

A **correlation** near +1 means "goes up together", near -1 means "one goes up, the other goes down", near 0 means "no straight-line relationship".

> **Careful:** correlation is not causation. Rain does not *force* people to buy chai. But it is a useful clue, and clues are what models learn from.

## 3. matplotlib: see it

Never trust numbers you have not plotted.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: rain vs chai, coloured by exam week
for flag, colour, label in [(False, "#4285F4", "normal week"), (True, "#EA4335", "exam week")]:
    part = df[df["exam_week"] == flag]
    axes[0].scatter(part["rain_mm"], part["chai_cups"], c=colour, label=label, alpha=0.75)
axes[0].set_xlabel("rain (mm)")
axes[0].set_ylabel("cups of chai sold")
axes[0].set_title("More rain, more chai")
axes[0].legend()

# Right: how are daily sales spread out?
axes[1].hist(df["chai_cups"], bins=20, color="#34A853", edgecolor="white")
axes[1].set_xlabel("cups of chai sold")
axes[1].set_ylabel("number of days")
axes[1].set_title("Daily sales")

plt.tight_layout()
plt.show()

## 4. A first taste of "learning"

Here is a model so simple you can read it in one line: draw the best straight line through *rain vs chai*. NumPy can find it for us.

In [ ]:
slope, intercept = np.polyfit(df["rain_mm"], df["chai_cups"], deg=1)
print(f"chai ≈ {intercept:.0f} + {slope:.1f} × rain_mm")
print("We built in 3.2 cups per mm of rain. The line found roughly that from noisy data.")

plt.figure(figsize=(6, 4))
plt.scatter(df["rain_mm"], df["chai_cups"], alpha=0.5, color="#4285F4")
xs = np.linspace(0, df["rain_mm"].max(), 50)
plt.plot(xs, intercept + slope * xs, color="#EA4335", linewidth=2)
plt.xlabel("rain (mm)")
plt.ylabel("cups of chai sold")
plt.title("A straight line is a model")
plt.show()

That straight line is machine learning in its simplest form: **find the pattern in past data, then use it to predict.** Next week you will do this properly with scikit-learn.

## 🏁 Take-home challenge

Use the empty cells below. Ideas:

1. Add a new column `hot_day` that is `True` when `temperature_c > 30`. Does chai differ on hot days?
2. Plot `temperature_c` against `chai_cups`. What do you notice?
3. Find the **three rainiest days**. How many cups did they sell?
4. Stretch: change the secret rule (for example `+ 45 * exam_week` to `+ 90 * exam_week`), re-run everything, and see if the data tells you.
5. Bring your own question. Think of a small dataset from campus life, write down the columns you *would* collect, and share it in the track channel.

When you finish, save a copy to your own GitHub (or Drive) and add what you learned to your Member Roadmap issue.

In [ ]:
# Your code here